# Metadata from SteamSpy

### General info and methodology

https://steamspy.com/api.php

SteamSpy has information about all appids, not only games.
So we will first retrieve the list of game_appids that we obtained previously from the steam api.
Then we will query the steamspy api and store json files with info about games only.

Saved json files of the form ``{appids}__{game name}__steamspy.json``

Stored in ``steamspy_dataset/``

Games' names that are too long (more thant 150 char) have been truncated.

```
 ## Return format for an app: ##

  * appid - Steam Application ID. If it's 999999, then data for this application is hidden on developer's request, sorry.
  * name - game's name
  * developer - comma separated list of the developers of the game
  * publisher - comma separated list of the publishers of the game
  * score_rank - score rank of the game based on user reviews
  * owners - owners of this application on Steam as a range.
  * average_forever - average playtime since March 2009. In minutes.
  * average_2weeks - average playtime in the last two weeks. In minutes.
  * median_forever - median playtime since March 2009. In minutes.
  * median_2weeks - median playtime in the last two weeks. In minutes.
  * ccu - peak CCU yesterday.
  * price - current US price in cents.
  * initialprice - original US price in cents.
  * discount - current discount in percents.
  * tags - game's tags with votes in JSON array.
  * languages - list of supported languages.
  * genre - list of genres.```


### Discrepancies in the number of retrieved games

We retrieved the information of **78,192** games from SteamSpy.
Meaning that we have no information about 67,430 games via the _request:all_.
Yet, when querying the information of one of these uncovered games in SteamSpy, e.g., https://steamspy.com/api.php?request=appdetails&appid=1675080 we have an answer.

According to Steam Spy documentation:
```
  ### all ###

  Returns all games with owners data sorted by owners. Returns 1,000 entries per page.
  * page - page number for the list (starts at 0)
```

It thus suggests that 67,430 games have no owners? Or that SteamSpy did not retrieve owner info for these games.


It can be that the game is not played or that it is not released yet.

In [13]:
import os
import json

game_appids = set()

for filename in os.listdir("./raw_metadata_dataset/"):
    if filename.endswith(".json") and "__" in filename:
        appid = filename.split("__")[0]
        if appid.isdigit():
            game_appids.add(appid)

print(f"# appids corresponding to games: {len(game_appids)}")


# appids corresponding to games: 145622.


In [ ]:
import requests

output_dir="steamspy_dataset"

def fetch_steamspy_data(page):
    base_url = "https://steamspy.com/api.php"
    all_games = {}

    params = {
        'request': 'all',
        'page': page
    }

    print(f"Fetching page {page}...")
    response = requests.get(base_url, params=params)
    if response.status_code != 200:
        print(f"Error fetching page {page}: Status {response.status_code}")
        return None

    data = response.json()
    if not data:
        print("No more data returned. Exiting.")
        return None

    for appid, game_info in data.items():
        if appid not in game_appids:
            continue

        game_name = game_info.get("name").strip()
        safe_name = game_name[:150]

        filename = f"{appid}__{safe_name}__steamspy.json"
        filepath = os.path.join(output_dir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(game_info, f, indent=2, ensure_ascii=False)

        all_games[appid] = game_info

    return all_games

page = 0
while True:
    games_data = fetch_steamspy_data(page)
    if games_data:
        print(f"Total games fetched: {len(games_data)}")
    else:
        break
    page += 1

In [17]:
steamspy_appids = set()
for filename in os.listdir("./steamspy_dataset/"):
    if filename.endswith(".json") and "__" in filename:
        appid = filename.split("__")[0]
        if appid.isdigit():
            steamspy_appids.add(appid)
print(f"Processed {len(steamspy_appids)} games from SteamSpy.")

remaining_games = game_appids - steamspy_appids
print("Games with no \"owners\" data in Steamspy: " + str(len(remaining_games)))

rg_list = list(remaining_games)
print("  E.g.,: " + str(rg_list[0:5]))


Processed 78192 games from SteamSpy.
Games with no "owners" data in Steamspy: 67430
  E.g.,: ['1555940', '2012270', '2984630', '1676330', '1675080']
